In [1]:
# Pregunta: ¿Cómo deben asignarse 40 cupos de tutoría para maximizar la ganancia esperada sin incumplir el mínimo por grado?

from pathlib import Path
import sys

import pandas as pd

professor_dir = Path.cwd().resolve()
activity_dir = professor_dir.parent
sys.path.insert(0, str(professor_dir))

from main import build_policy_contract, build_recommendation, evaluate_policies, explain_selection, select_policy


In [2]:
policies = pd.read_csv(activity_dir / 'data' / 'policy_options.csv')
policies[['policy', 'grade_1_slots', 'grade_2_slots', 'grade_3_slots', 'total_slots', 'min_slots_per_grade']]


,policy,grade_1_slots,grade_2_slots,grade_3_slots,total_slots,min_slots_per_grade
0,reparto_igualitario,14,13,13,40,8
1,solo_mayor_beneficio,8,24,8,40,8
2,prioridad_con_piso,12,20,8,40,8


In [3]:
# La comparación hace visible qué políticas son factibles antes de valorar su beneficio esperado.

assessment = evaluate_policies(policies)
assessment[['policy', 'allocated_slots', 'expected_learning_gain', 'capacity_ok', 'minimum_ok', 'feasible']]


,policy,allocated_slots,expected_learning_gain,capacity_ok,minimum_ok,feasible
0,reparto_igualitario,40,4930,True,True,True
1,solo_mayor_beneficio,40,5520,True,True,True
2,prioridad_con_piso,40,5360,True,True,True


In [4]:
selected = select_policy(assessment)
explained_assessment = explain_selection(assessment, selected['policy_id'])
explained_assessment[['policy', 'expected_learning_gain', 'decision_status', 'decision_reason']]


,policy,expected_learning_gain,decision_status,decision_reason
0,reparto_igualitario,4930,descartada,Ofrece menor ganancia esperada que la política...
1,solo_mayor_beneficio,5520,seleccionada,Maximiza la ganancia esperada entre las políti...
2,prioridad_con_piso,5360,descartada,Ofrece menor ganancia esperada que la política...


In [5]:
import json

submission_dir = activity_dir / 'submission'
policy_contract = build_policy_contract(selected, explained_assessment)

explained_assessment.to_csv(submission_dir / 'policy_assessment.csv', index=False)
build_recommendation(selected).to_csv(submission_dir / 'policy_recommendation.csv', index=False)
(submission_dir / 'policy_contract.json').write_text(
    json.dumps(policy_contract, ensure_ascii=False, indent=2) + '\n',
    encoding='utf-8',
)

policy_contract


{'analytical_question': '¿Cómo deben asignarse 40 cupos de tutoría para maximizar la ganancia esperada sin incumplir el mínimo por grado?',
 'decision_context': 'Asignación de cupos de tutoría antes de cada periodo académico.',
 'decision_cadence': 'Una vez por periodo académico, antes de abrir las tutorías.',
 'response_need': 'La asignación debe estar aprobada antes de la matrícula del periodo.',
 'decision_owner': 'Dirección académica',
 'execution_mode': 'Aprobación humana de una recomendación computable.',
 'selected_policy': {'policy_id': 'P1',
  'policy': 'solo_mayor_beneficio',
  'grade_1_slots': 8,
  'grade_2_slots': 24,
  'grade_3_slots': 8},
 'objective': 'Maximizar la ganancia esperada de aprendizaje.',
 'expected_learning_gain': 5520,
 'constraints': {'total_slots': 40, 'minimum_slots_per_grade': 8},
 'safeguards': ['No asignar más cupos que la capacidad disponible.',
  'Garantizar el mínimo de cupos definido para cada grado.'],
 'exception_and_human_authority': 'Escalar a